In [1]:
import numpy as np
import scipy.linalg as la
from qiskit.quantum_info import SparsePauliOp
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeMontrealV2
from qiskit_aer.noise import NoiseModel
import warnings, time
warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


In [2]:
N_QUBITS       = 8
DIM            = 2 ** N_QUBITS
MAX_ITERATIONS = 40
VD_COPIES      = 2
N_SEEDS        = 5
REG_LAMBDA     = 1e-3

_I = np.eye(2, dtype=complex)
_X = np.array([[0,1],[1,0]], dtype=complex)
_Y = np.array([[0,-1j],[1j,0]], dtype=complex)
_Z = np.array([[1,0],[0,-1]], dtype=complex)

_PAIRS_2Q = {
    'XX': np.kron(_X,_X), 'XY': np.kron(_X,_Y), 'XZ': np.kron(_X,_Z),
    'YX': np.kron(_Y,_X), 'YY': np.kron(_Y,_Y), 'YZ': np.kron(_Y,_Z),
    'ZX': np.kron(_Z,_X), 'ZY': np.kron(_Z,_Y), 'ZZ': np.kron(_Z,_Z),
}

_NN_PAULI_NAMES = ['XX','YY','ZZ']
N_PAIRS  = N_QUBITS - 1
N_TERMS  = N_PAIRS * len(_NN_PAULI_NAMES)   # 21
N_PARAMS = 2 * N_TERMS                       # 42

_TERM_INFO = []
for _pi in range(N_PAIRS):
    for _pn in _NN_PAULI_NAMES:
        _TERM_INFO.append({
            'P2q'   : _PAIRS_2Q[_pn],
            'numpy_i': _pi,
            'numpy_j': _pi + 1,
            'qk_qa' : N_QUBITS - 2 - _pi,
            'qk_qb' : N_QUBITS - 1 - _pi,
        })

_PERM = [int(format(i, f'0{N_QUBITS}b')[::-1], 2) for i in range(DIM)]
print(f'N_TERMS={N_TERMS}  N_PARAMS={N_PARAMS}  Constants OK')

N_TERMS=21  N_PARAMS=42  Constants OK


In [3]:
print('Setting up FakeMontrealV2 ...')
_fake_backend = FakeMontrealV2()
_noise_model  = NoiseModel.from_backend(_fake_backend)
_sim          = AerSimulator(noise_model=_noise_model, method='density_matrix')
print('Ready: FakeMontrealV2 (27-qubit IBM device noise)')

Setting up FakeMontrealV2 ...
Ready: FakeMontrealV2 (27-qubit IBM device noise)


In [4]:
def _qiskit_dm_to_numpy(rho_q):
    return rho_q[np.ix_(_PERM, _PERM)]

def _apply_gate(U_2q, psi_tensor, qi, qj):
    N = psi_tensor.ndim
    axes = [qi, qj] + [k for k in range(N) if k != qi and k != qj]
    psi  = np.transpose(psi_tensor, axes).reshape(4, -1)
    psi  = (U_2q @ psi).reshape((2, 2) + (2,)*(N-2))
    return np.transpose(psi, np.argsort(axes))

def compute_ideal_state(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    psi   = np.zeros((2,)*N_QUBITS, dtype=complex)
    psi[(0,)*N_QUBITS] = 1.0
    for j, t in enumerate(_TERM_INFO):
        psi = _apply_gate(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']), psi, t['numpy_i'], t['numpy_j'])
    return psi.flatten()

def _upte_reg_penalty(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    total = 0.0
    for _pi in range(N_PAIRS):
        B = np.zeros((4,4), dtype=complex)
        for _pp, _pn in enumerate(_NN_PAULI_NAMES):
            j = _pi * len(_NN_PAULI_NAMES) + _pp
            B += rho_n[j] * la.expm(-1j * rho_n[j] * tau[j] * _PAIRS_2Q[_pn])
        dev = B.conj().T @ B - np.eye(4, dtype=complex)
        total += np.real(np.trace(dev.conj().T @ dev))
    return total

def apply_fake_backend_noise(params):
    rho_w = np.maximum(params[:N_TERMS], 1e-6)
    rho_n = rho_w / rho_w.sum()
    tau   = params[N_TERMS:]
    qc = QuantumCircuit(N_QUBITS)
    for j, t in enumerate(_TERM_INFO):
        qc.unitary(la.expm(-1j * rho_n[j] * tau[j] * t['P2q']), [t['qk_qa'], t['qk_qb']])
    qc.save_density_matrix()
    qc_t   = transpile(qc, _sim, optimization_level=1)
    result = _sim.run(qc_t, shots=1).result()
    return _qiskit_dm_to_numpy(np.array(result.data(0)['density_matrix']))

def noise_aware_objective(params, psi_target):
    psi_ideal = compute_ideal_state(params)
    rho_noisy = apply_fake_backend_noise(params)
    ideal_fid = float(np.abs(np.vdot(psi_target, psi_ideal))**2)
    noisy_fid = float(np.clip(np.real(psi_target.conj() @ rho_noisy @ psi_target), 0, 1))
    loss      = 1.0 - noisy_fid + REG_LAMBDA * _upte_reg_penalty(params)
    return loss, noisy_fid, ideal_fid, rho_noisy

def virtual_distillation(rho, psi_target, n=2):
    rhoM = rho.copy()
    for _ in range(n-1): rhoM = rhoM @ rho
    tr = np.trace(rhoM)
    if abs(tr) < 1e-12: return 0.0
    return float(np.clip(np.real(psi_target.conj() @ (rhoM/tr) @ psi_target), 0, 1))

print('Core functions OK')

Core functions OK


In [5]:
class MMAOptimizer:
    def __init__(self, n, move_limit=0.4, gamma=0.5):
        self.n, self.move_limit, self.gamma = n, move_limit, gamma
        self.L = self.U = self.prev_params = self.prev_loss = None
    def init(self, p0, delta=0.6):
        self.L, self.U = p0 - delta, p0 + delta
        self.prev_params = p0.copy()
    def step(self, x, grad):
        x_new = np.zeros_like(x)
        for i in range(self.n):
            g, xi, Li, Ui = grad[i], x[i], self.L[i], self.U[i]
            pi = abs(g)*(Ui-xi)**2 if g < 0 else 0.0
            qi = abs(g)*(xi-Li)**2 if g >= 0 else 0.0
            dn = pi/(Ui-xi+1e-12)**2 + qi/(xi-Li+1e-12)**2
            if dn > 1e-12:
                x_new[i] = xi + (pi/(Ui-xi+1e-12) - qi/(xi-Li+1e-12)) / dn
            else:
                x_new[i] = xi
            x_new[i] = np.clip(x_new[i], max(xi-self.move_limit, Li+1e-6), min(xi+self.move_limit, Ui-1e-6))
        return x_new
    def update(self, x, loss):
        good = self.prev_loss is None or loss < self.prev_loss - 1e-8
        s = 1.2/self.gamma if good else self.gamma
        self.L = x - s*(x - self.L);  self.U = x + s*(self.U - x)
        self.L = np.minimum(self.L, self.U - 1e-4)
        self.U = np.maximum(self.U, self.L + 1e-4)
        self.prev_params = x.copy();  self.prev_loss = loss

_prev_grad = None

def hybrid_gradient(params, psi_target):
    global _prev_grad
    grad = np.zeros_like(params)
    for i in range(N_TERMS):
        pp = np.clip(params.copy(), 1e-6, None); pp[i] += 1e-4
        pm = np.clip(params.copy(), 1e-6, None); pm[i] -= 1e-4
        grad[i] = (noise_aware_objective(pp, psi_target)[0] - noise_aware_objective(pm, psi_target)[0]) / 2e-4
    for i in range(N_TERMS, N_PARAMS):
        pp = params.copy(); pp[i] += np.pi/4
        pm = params.copy(); pm[i] -= np.pi/4
        grad[i] = (noise_aware_objective(pp, psi_target)[0] - noise_aware_objective(pm, psi_target)[0]) / (np.pi/2)
    if _prev_grad is None:
        _prev_grad = grad.copy(); return grad
    g = 0.4*grad + 0.6*_prev_grad
    _prev_grad = g.copy(); return g

def run_dual_mma(init_params, psi_target, max_iter=MAX_ITERATIONS):
    global _prev_grad
    _prev_grad = None
    mma_r = MMAOptimizer(N_TERMS, move_limit=0.2); mma_r.init(init_params[:N_TERMS], delta=0.4)
    mma_t = MMAOptimizer(N_TERMS, move_limit=0.6); mma_t.init(init_params[N_TERMS:], delta=0.8)
    cur = init_params.copy()
    cur_loss, cur_fid, cur_ifid, _ = noise_aware_objective(cur, psi_target)
    print(f'  Initial: NoisyFid={cur_fid:.4f}  IdealFid={cur_ifid:.4f}')
    best_fid, best_params, stag = cur_fid, cur.copy(), 0
    for it in range(max_iter):
        grad  = hybrid_gradient(cur, psi_target)
        new   = np.concatenate([mma_r.step(cur[:N_TERMS], grad[:N_TERMS]),
                                 mma_t.step(cur[N_TERMS:], grad[N_TERMS:])])
        new_loss, new_fid, _, _ = noise_aware_objective(new, psi_target)
        delta = new_fid - cur_fid
        if new_loss < cur_loss - 1e-6 or delta > -1e-5:
            cur, cur_loss, cur_fid = new, new_loss, new_fid
            if cur_fid > best_fid: best_fid, best_params, stag = cur_fid, cur.copy(), 0
            else: stag += 1
            mma_r.move_limit = min(0.4, mma_r.move_limit*(1.4 if delta>0.01 else 1.2 if delta>0.001 else 0.9))
        else:
            mma_r.move_limit = max(0.02, mma_r.move_limit*0.8); stag += 1
        mma_t.move_limit = mma_r.move_limit * 2.0
        mma_r.update(cur[:N_TERMS], cur_loss)
        mma_t.update(cur[N_TERMS:], cur_loss)
        if it % 5 == 0 or it == max_iter-1:
            print(f'  Iter {it+1:2d}: NoisyFid={cur_fid:.4f} (Δ={delta:+.4f})')
        if stag > 15: print(f'  Stagnated at iter {it+1}.'); break
    print(f'  Best NoisyFid: {best_fid:.4f}')
    return best_params, best_fid

def build_target(seed=None):
    rng = np.random.default_rng(seed)
    terms = []
    for i in range(N_QUBITS-1):
        for p in ['XX','YY','ZZ']:
            s = ['I']*N_QUBITS; s[i]=p[0]; s[i+1]=p[1]; terms.append(''.join(s))
    return la.expm(-1j * SparsePauliOp(terms, coeffs=rng.uniform(-0.5,0.5,len(terms))).to_matrix(sparse=False))

print('Optimizer + helpers OK')

Optimizer + helpers OK


In [ ]:
U_target   = build_target(seed=0)
psi_0      = np.zeros(DIM, dtype=complex); psi_0[0] = 1.0
psi_target = U_target @ psi_0

all_records = []
for seed in range(N_SEEDS):
    print(f"\n{'='*55}")
    print(f"SEED {seed+1}/{N_SEEDS}")
    print('='*55)
    rng = np.random.default_rng(seed + 42)
    init_params = np.concatenate([
        rng.uniform(0.1, 0.4, N_TERMS),
        rng.uniform(0.1, np.pi, N_TERMS),
    ])
    _, init_noisy, init_ideal, init_rho = noise_aware_objective(init_params, psi_target)
    init_vd = virtual_distillation(init_rho, psi_target, VD_COPIES)
    print(f'  Pre-opt:  ideal={init_ideal:.4f}  noisy={init_noisy:.4f}  vd={init_vd:.4f}')

    t0 = time.time()
    best_params, _ = run_dual_mma(init_params, psi_target, MAX_ITERATIONS)
    elapsed = time.time() - t0

    _, final_noisy, final_ideal, final_rho = noise_aware_objective(best_params, psi_target)
    final_vd = virtual_distillation(final_rho, psi_target, VD_COPIES)
    rec = dict(seed=seed, init_noisy=init_noisy, final_noisy=final_noisy,
               final_vd=final_vd, improvement=final_noisy-init_noisy, elapsed=elapsed)
    all_records.append(rec)
    print(f'  Final:  noisy={final_noisy:.4f}  vd={final_vd:.4f}  Δ={rec["improvement"]:+.4f}  t={elapsed:.0f}s')

print(f"\n{'='*55}")
print('MONTE CARLO SUMMARY')
print('='*55)
noisy = [r['final_noisy'] for r in all_records]
vd    = [r['final_vd']    for r in all_records]
impr  = [r['improvement'] for r in all_records]
print(f'NoisyFid  mean={np.mean(noisy):.4f} ± {np.std(noisy):.4f}  best={max(noisy):.4f}')
print(f'VD Fid    mean={np.mean(vd):.4f} ± {np.std(vd):.4f}')
print(f'Improvement mean={np.mean(impr):+.4f} ± {np.std(impr):.4f}')
print(f'Converged: {sum(r["improvement"]>0 for r in all_records)}/{N_SEEDS}')
print(f'Total runtime: {sum(r["elapsed"] for r in all_records)/60:.1f} min')


SEED 1/5
  Pre-opt:  ideal=0.0894  noisy=0.0743  vd=0.0998
  Initial: NoisyFid=0.0743  IdealFid=0.0894
  Iter  1: NoisyFid=0.1430 (Δ=+0.0687)
  Iter  6: NoisyFid=0.5113 (Δ=-0.1178)
  Iter 11: NoisyFid=0.5342 (Δ=+0.0032)
  Iter 16: NoisyFid=0.5377 (Δ=-0.0031)
  Iter 21: NoisyFid=0.5406 (Δ=+0.0013)
  Iter 26: NoisyFid=0.5421 (Δ=+0.0007)
  Iter 31: NoisyFid=0.5423 (Δ=+0.0001)
  Iter 36: NoisyFid=0.5423 (Δ=-0.0001)
  Iter 40: NoisyFid=0.5423 (Δ=+0.0000)
  Best NoisyFid: 0.5423
  Final:  noisy=0.5423  vd=0.5792  Δ=+0.4680  t=734s

SEED 2/5
  Pre-opt:  ideal=0.0832  noisy=0.0707  vd=0.0947
  Initial: NoisyFid=0.0707  IdealFid=0.0832
  Iter  1: NoisyFid=0.1767 (Δ=+0.1060)
  Iter  6: NoisyFid=0.4565 (Δ=-0.0047)
  Iter 11: NoisyFid=0.5505 (Δ=-0.0263)
  Iter 16: NoisyFid=0.5700 (Δ=-0.0314)
  Iter 21: NoisyFid=0.5722 (Δ=+0.0000)
  Iter 26: NoisyFid=0.5722 (Δ=-0.0006)
  Iter 31: NoisyFid=0.5722 (Δ=-0.0002)
  Iter 36: NoisyFid=0.5722 (Δ=-0.0000)
  Iter 40: NoisyFid=0.5722 (Δ=+0.0000)
  Best NoisyF